##**<font color=blue>Problem Statement**

### Business Context

NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. Traditional search methods and generic news feeds often lead to information overload and inefficiencies, making it challenging for users to access relevant and trustworthy content efficiently.

To address this, NewsFindr wants to leverage Agentic AI to build an AI-powered news retrieval agent that ensures accuracy and credibility. By utilizing a structured, multi-step approach, the system will provide secure, fair, and explainable recommendations - enhancing user engagement, optimizing content discovery, and improving access to timely and relevant news.

### Objective

- Provide real-time, personalized news retrieval to help users discover relevant
content effortlessly.

- Ensure accuracy and credibility by sourcing news from trusted platforms and minimizing misinformation.

- Improve user engagement through seamless content discovery, reducing information overload.

- Streamline the news consumption process by eliminating outdated and irrelevant content, providing a refined reading experience.

##**<font color=blue>Installing and Importing Necessary Libraries and Dependencies**

In [1]:
# Install/upgrade required packages
%pip install -qU pip
%pip install -q pandas numpy==1.26.4 scipy python-dotenv ddgs
%pip install -q langchain==0.2.17 langchain-core==0.2.43 langchain-community==0.2.19 langchain-groq==0.1.10 langchain-huggingface==0.0.3 langchain-text-splitters==0.2.4 langchainhub==0.1.20 langchain-experimental==0.0.62

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


**Note:**

- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [2]:
import json
import os
import pandas as pd
import sqlite3

from langchain.agents import create_sql_agent, initialize_agent 
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.agents.agent_types import AgentType
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain import hub
from langchain.agents import load_tools
from langchain.agents import Tool

from getpass import getpass
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field, ValidationError
from dotenv import load_dotenv

from ddgs import DDGS
from typing import List, Optional, Dict

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

### Creating Groq API Key

In [3]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
print("Loaded" if groq_api_key else "Missing")

Loaded


Initializes Gorq model with different temperature settings.

In [4]:

llm_high = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.9,
    groq_api_key=groq_api_key
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    groq_api_key=groq_api_key,
    max_tokens=180
)

print("Groq models initialized successfully")

Groq models initialized successfully


##**<font color=blue>Checking LLMs response**

A low temperature setting ensures consistent and repeatable responses, making the output more deterministic. It is ideal for tasks requiring accuracy, such as fact-based queries, structured data extraction, and rule-based decision-making.

In [5]:
# Deterministic response check
for i in range(5):
    response = llm.predict("What is the capital of India? Reply in one line.")
    print(f"Run {i+1}: {response}")

Run 1: The capital of India is New Delhi.
Run 2: The capital of India is New Delhi.
Run 3: The capital of India is New Delhi.
Run 4: The capital of India is New Delhi.
Run 5: The capital of India is New Delhi.


A high temperature setting introduces more randomness and variability in responses, making the output creative and diverse. It is useful for tasks like brainstorming, storytelling, content generation, and exploring multiple perspectives.

In [6]:
# Creative response check
for i in range(5):
    response = llm_high.predict("Write a short, catchy one-line headline about AI in news discovery.")
    print(f"Run {i+1}: {response}")

Run 1: "Revolutionizing the News Feed: AI-Powered Discovery Takes Center Stage, Bringing You the Stories That Matter Most."
Run 2: "AI Revolutionizes News Discovery: Uncovering Hidden Gems with Intelligent Insights"
Run 3: "Revolutionizing the Newsfeed: AI Powers Personalized Story Discovery for a Smarter You"
Run 4: "Revolutionizing the newsfeed: AI-powered discovery redefines the way you stay informed, one headline at a time."
Run 5: "Revolutionizing the News: AI-Powered Discovery Unleashes a Smarter Way to Stay Informed"


##<font color=blue>**SQL Agent for Data Retrieval and Customer email verification and reterival**

Facilitates efficient data retrieval and modification, enabling seamless customer information management through SQL interactions

<font color=magenta>**Function setup_sql_agent**</FONT>: Initializes an AI-powered SQL agent for querying and updating a database.
It connects to an SQL database, and sets up an agent and check for the presence of the email in the DB, if not found return as "email not found", if email found return with the "name of the customer and area of interest".

In [7]:
# SQL Agent for Data Retrieval 

import sqlite3
import json
from typing import Dict, Any

DB_PATH = "////root/ashiravi/customer.db"  

# System message defines behavior rules for our SQL retrieval agent
system_message = """
You are a SQL retrieval assistant for customer verification.
Rules:
1. Use only the customers table from customer.db
2. If email is not found, return exactly: email not found
3. If email is found, return customer name and interests
4. Do not fabricate data
5. Keep responses concise and structured
""".strip()

print("System message initialized.")
print(system_message)

System message initialized.
You are a SQL retrieval assistant for customer verification.
Rules:
1. Use only the customers table from customer.db
2. If email is not found, return exactly: email not found
3. If email is found, return customer name and interests
4. Do not fabricate data
5. Keep responses concise and structured


In [8]:
class SQLiteCustomerAgent:
    """
    Lightweight SQL agent using sqlite3 for reliable data retrieval.
    Supports:
    - listing unique emails
    - verifying email existence
    - retrieving customer details (name, interests)
    """
    def __init__(self, db_path: str, system_message: str):
        self.db_path = db_path
        self.system_message = system_message

    def _connect(self):
        return sqlite3.connect(self.db_path)

    def fetch_unique_emails(self) -> Dict[str, Any]:
        query = "SELECT DISTINCT email FROM customers ORDER BY email;"
        with self._connect() as conn:
            cur = conn.cursor()
            cur.execute(query)
            rows = cur.fetchall()

        emails = [r[0] for r in rows]
        return {
            "status": "success",
            "count": len(emails),
            "emails": emails
        }

    def verify_email_and_get_details(self, email: str) -> Dict[str, Any]:
        query = """
        SELECT name, email, interests, last_updated
        FROM customers
        WHERE email = ?
        LIMIT 1;
        """
        with self._connect() as conn:
            cur = conn.cursor()
            cur.execute(query, (email,))
            row = cur.fetchone()

        if row is None:
            return {
                "status": "not_found",
                "message": "email not found"
            }

        name, email_value, interests_raw, last_updated = row

        # interests stored as JSON-like string in DB
        try:
            interests = json.loads(interests_raw) if interests_raw else []
        except Exception:
            # fallback if malformed
            interests = [interests_raw]

        return {
            "status": "found",
            "name": name,
            "email": email_value,
            "interests": interests,
            "last_updated": last_updated
        }

In [9]:
# Initialize SQL agent
sql_agent = SQLiteCustomerAgent(DB_PATH, system_message)
print("SQL agent initialized and ready.")

SQL agent initialized and ready.


In [10]:
unique_email_result = sql_agent.fetch_unique_emails()

print("=== Unique Emails ===")
print(f"Total unique emails: {unique_email_result['count']}")
for e in unique_email_result["emails"]:
    print("-", e)

=== Unique Emails ===
Total unique emails: 15
- alice.6eb33c45-5@gmail.com
- alice.d56e53b6-1@gmail.com
- emma.a88fec03-c@gmail.com
- george.5483cb53-8@gmail.com
- hannah.4770b814-2@gmail.com
- ian.203631a0-b@gmail.com
- ian.aee37571-7@gmail.com
- julia.b5a14512-c@gmail.com
- julia.cf4d1edb-9@gmail.com
- julia.d77d96f3-3@gmail.com
- kevin.f8641860-7@gmail.com
- lily.45e01919-0@gmail.com
- lily.fa8fe05b-7@gmail.com
- nora.c63fa237-e@gmail.com
- oscar.edd38e10-6@gmail.com


In [11]:
email_not_present = "testtesttest@gmail.com"
result_not_present = sql_agent.verify_email_and_get_details(email_not_present)

print("\n=== Email Verification (Not Present) ===")
print("Input Email:", email_not_present)
print("Output:", result_not_present["message"])


=== Email Verification (Not Present) ===
Input Email: testtesttest@gmail.com
Output: email not found


In [12]:
email_present = "alice.d56e53b6-1@gmail.com"  
result_present = sql_agent.verify_email_and_get_details(email_present)

print("\n=== Email Verification (Present) ===")
print("Input Email:", email_present)
print("Status:", result_present["status"])
print("Name:", result_present["name"])
print("Email:", result_present["email"])
print("Interests:", result_present["interests"])
print("Last Updated:", result_present["last_updated"])


=== Email Verification (Present) ===
Input Email: alice.d56e53b6-1@gmail.com
Status: found
Name: Alice
Email: alice.d56e53b6-1@gmail.com
Interests: ['International', 'Technology']
Last Updated: 2025-03-26 06:46:15


##<font color=blue>**Generating Expanded Search Queries for Current News**

Generating Expanded Search Queries for Current News
This function, expand_search_queries, generates precise and up-to-date search queries based on the user's interests. It ensures that each query is crafted to retrieve breaking news, trending topics, or recent developments without relying on specific years.

The function takes user interests and User_query as inputs and processes  through an LLM  to generate a relevant search query.

A system prompt guides the model to focus only on current events and avoid outdated information.

The expanded search queries are stored in expanded_search and returned for later use in retrieving news articles.

In [13]:
def expand_search_queries(inputs) -> list:
    """
    Expand user interests into time-sensitive search queries for breaking news.
    Accepts either a dict (with 'interests' and 'user_query') or a raw string.
    """
    # Handle dict input
    if isinstance(inputs, dict):
        interests = inputs.get("interests", [])
        user_query = inputs.get("user_query","")
    else:
        # Handle string input fallback
        interests = [i.strip() for i in str(inputs).split(",")]
        user_query = ""

    system_prompt = """You are a news query expansion assistant.
Generate exactly ONE search query per interest.
CRITICAL:
- Prioritize the user's specific query intent over generic interest terms.
- If user query mentions startups or travel, include those exact terms.
- Avoid unrelated political or election terms unless explicitly asked.
- Focus on latest/current updates.
- Return only plain query text."""                                                                            

    expanded_queries = []
    for interest in interests:
        prompt = f"Generate one search query related to: '{interest}' considering the user query: '{user_query}'"
        response = llm.predict_messages(
            [
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ]
        )
        query = response.content.strip()
        if query:
            expanded_queries.append(query)

    return expanded_queries

# Define the tool
expand_tool = Tool(
    name="ExpandSearchQueries",
    func=expand_search_queries,
    description="Expand interests into latest news queries."
)

##<font color=blue>**Fetch News Results Using DuckDuckGo**

This function retrieves news results from DuckDuckGo based on the user's expanded search queries.

In [14]:
def ddg_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=3):                              
            title = r.get("title", "")
            url = r.get("href", "")
            body = r.get("body", "")
            res={"title":title,"url":url,"body":body}
            results.append(res)
    return results

ddg_search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Fetch top recent search results."
)

##<font color=blue>**Filter Relevant and Trustworthy URLs Based on User Interests**

This function filters search result URLs, assesses the credibility of news sources, and retains only those that are relevant to the user's interests.

Extracts URLs and their body content from the search results.

Updates the state with the refined list of relevant URLs and their descriptions.

In [15]:
def filter_with_llm(search_results) -> list:
    # Normalize input to list
    if isinstance(search_results, str):
        try:
            parsed = json.loads(search_results)
            search_results = parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            # If plain text, wrap into one pseudo-item
            search_results = [{"title": "", "url": "", "body": search_results}]
    elif isinstance(search_results, dict):
        search_results = [search_results]
    elif not isinstance(search_results, list):
        search_results = [{"title": "", "url": "", "body": str(search_results)}]

    # Compact payload (token-safe)
    compact = []
    for r in search_results[:8]:
        if isinstance(r, dict):
            compact.append({
                "title": str(r.get("title", ""))[:120],
                "url": str(r.get("url", ""))[:200],
                "body": str(r.get("body", ""))[:220]
            })
        else:
            compact.append({
                "title": "",
                "url": "",
                "body": str(r)[:220]
            })

    search_results_json = json.dumps(compact, indent=2)

    system_prompt = """You are a news credibility evaluator.
Return ONLY a JSON array of relevant and trustworthy items.
Each item must include: title, url, body.
Return at most 3 items."""

    prompt = "Filter these results:\n" + search_results_json

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )
    return response.content

credibility_tool = Tool(
    name="CredibilityFilter",
    func=filter_with_llm,
    description="Filter relevant trustworthy URLs."
)

##<FONT COLOR=BLUE>**Generate summary for the URLs**

This function generates summaries for the credible URLs

Returns the URLs and the Summary based on the users interest

In [16]:
def summarize_news(url_list) -> dict:
    # normalize input -> only URLs
    if isinstance(url_list, str):
        try:
            url_list = json.loads(url_list)
        except:
            url_list = [url_list]

    urls = []
    for x in url_list:
        if isinstance(x, dict) and x.get("url"):
            urls.append(x["url"])
        elif isinstance(x, str):
            urls.append(x)

    urls = urls[:3]  # hard cap

    system_prompt = """Summarize latest key points from these URLs briefly and factually."""
    prompt = "Summarize these URLs:\n" + "\n".join(urls)

    response = llm.predict_messages(
        [SystemMessage(content=system_prompt), HumanMessage(content=prompt)]
    )
    return {"summary": response.content, "urls": urls}


summarize_tool = Tool(
    name="SummarizeNews",
    func=summarize_news,
    description="Summarize shortlisted URLs."
)


##<font color=blue>**Creating an Agent**

In [17]:
#Final code
tools = [expand_tool, ddg_search_tool, credibility_tool, summarize_tool]                                                    

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    max_iterations=25,
    early_stopping_method="generate",
    handle_parsing_errors=True
)

##<font color=blue>**Display Final Verified URLs and Summary based on the User interest and User query**

In [18]:
def get_customer_profile(email: str):
    """
    Wrapper function to retrieve customer profile by email.
    Returns {'status': 'not_found', 'message': 'email not found'} if absent.
    """
    return sql_agent.verify_email_and_get_details(email)

In [19]:
# Final code
email = "alice.d56e53b6-1@gmail.com"
profile = get_customer_profile(email)

if profile.get("status") == "not_found":
    print("email not found")
else:
    interests = profile.get("interests", [])
    user_query = "latest developments and key updates"

    # 1) Expand queries
    expanded = expand_search_queries({"interests": interests, "user_query": user_query})
    print("Expanded queries:", expanded)

    # 2) Search for all queries
    all_results = []
    for q in expanded:
        all_results.extend(ddg_search(q))

    # 3) Credibility filter
    filtered_raw = filter_with_llm(all_results)

    # Parse filtered output safely
    try:
        filtered = json.loads(filtered_raw) if isinstance(filtered_raw, str) else filtered_raw
    except Exception:
        filtered = all_results[:5]  # fallback

    # 4) Top 3 URLs
    top_urls = [item.get("url") for item in filtered if isinstance(item, dict) and item.get("url")][:3]

    # 5) Summarize
    summary_output = summarize_news(top_urls)

    print("\n======= FINAL RESPONSE =======")
    print("Verified URLs:")
    for u in top_urls:
        print("-", u)

    print("\nSummary:")
    print(summary_output["summary"])

Expanded queries: ['latest international news and current global developments', 'latest technology developments and current updates']

======= FINAL RESPONSE =======
Verified URLs:
- https://www.theguardian.com/world
- https://edition.cnn.com/
- https://www.reuters.com/world/

Summary:
Here's a brief summary of the latest news from the provided URLs:

1. **The Guardian (https://www.theguardian.com/world)**: The website features news on global events, including the ongoing conflict in Ukraine, climate change, and social issues. Recent articles cover topics such as:
	* International relations and diplomacy
	* Global health and pandemic updates
	* Environmental concerns and sustainability
2. **CNN (https://edition.cnn.com/)**: The website provides up-to-date news on global events, politics, business, and entertainment. Recent articles cover topics such as:
	* US politics and elections
	* Global conflicts and security issues
	* Business and economic news, including market trends and analys

##<font color=blue>**Retrieve URLs and Summaries for Three Areas of Interest**

In [20]:
def query_response(email: str, user_query: str) -> str:
    # Fetch interests based on the provided email
    profile = get_customer_profile(email)

    if profile.get("status") == "not_found":
        print("\n======= FINAL RESPONSE =======")
        print("email not found")
        return "email not found"

    interests = profile.get("interests", [])

    # Agent prompt stays the same
    agent_prompt = f"""
    The user's interest is: {interests} and specifically looking for {user_query}.
    Here is the process to follow:
    1. Expand the user's interest into news search queries using the 'ExpandSearchQueries' tool.
       The input to this tool should be a dictionary like: {{'interests': {interests}, 'user_query': '{user_query}'}}.
    2. Use the query generated in step 1 to search for recent news using the 'DuckDuckGoSearch' tool for all the queries.
    3. Pass the combined results of all queries into 'CredibilityFilter' for filtering.
    4. From the filtered results, select the top 3 latest news articles for summarization.
    5. Summarize all credible results into a detailed summary using the 'SummarizeNews'.
    6. Provide the final summary to the user.
    """

    # Add strict anti-loop instruction WITHOUT changing your main prompt content
    anti_loop_suffix = """
    Important execution constraints:
    - Call each tool at most once per step.
    - Do not call ExpandSearchQueries more than one time.
    - After getting expanded queries, proceed to DuckDuckGoSearch.
    - Return final answer within one pass.
    """

    try:
        response = agent.run(agent_prompt + anti_loop_suffix)
    except Exception:
        # Fallback: still use same prompt via invoke
        response = agent.invoke({"input": agent_prompt + anti_loop_suffix})
        response = response["output"] if isinstance(response, dict) and "output" in response else str(response)

    print("\n======= FINAL RESPONSE =======")
    print(response)
    return response

In [21]:
#Check for the user ID
email = "kevin.f8641860-7@gmail.com"                                                           
user_query = "latest updates in startups and travel trends"                                                         
response = query_response(email, user_query)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01km7xev1wejy8e069yh4nnc8z` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99648, Requested 963. Please try again in 8m47.904s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
#Check for the user ID
email = "julia.cf4d1edb-9@gmail.com"                                                             
user_query = "latest sports and entertainment highlights"                                                       
response = query_response(email, user_query)

In [ ]:
#Check for the user ID
email = "george.5483cb53-8@gmail.com"                                                             
user_query = "latest policy and science developments in India"                                                       
response = query_response(email, user_query)

##<font color=blue>**Conclusion**

1. The project successfully built a structured Agentic AI pipeline for personalized news discovery using customer interests from customer.db.
2. SQL retrieval and email verification worked correctly, including valid-user detail extraction and proper email not found handling.
3. The workflow of query expansion → DuckDuckGo retrieval → credibility filtering → summarization was implemented end-to-end and produced usable outputs.
4. Results were generally relevant and concise, showing reduced information overload and better targeted news recommendations.
5. Some responses were broad or missed credible sources for specific prompts, highlighting future scope to improve relevance ranking, source validation, and fallback handling.